# Model 1: Inference Only
**Österreichisches Steuerrecht — Zero-Shot Inference**

Modell: `Qwen/Qwen2.5-3B-Instruct`  
Herausragendes deutsches Sprachverständnis, ideal für Kaggle T4.


In [ ]:
# ZELLE 1: Pakete installieren (einmalig)
!pip install -q transformers torch accelerate

In [ ]:
# ZELLE 2: GPU-Check
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Gerät: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ZELLE 3: Pfade konfigurieren
import os
DATASET_PATH = '/kaggle/input/datasets/lucariggler/wu-steuerrecht/dataset_clean.csv'
OUTPUT_PATH = '/kaggle/working/model_1_output.csv'
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)


In [ ]:
# ZELLE 4: Modell laden
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, time
MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
model.eval()


In [ ]:
# ZELLE 5: Inferenz
import pandas as pd
from tqdm import tqdm
df = pd.read_csv(DATASET_PATH, encoding='utf-8-sig')
pd.DataFrame(columns=['id', 'answer']).to_csv(OUTPUT_PATH, index=False)
for i, row in tqdm(df.iterrows(), total=len(df)):
    messages = [
        {'role': 'system', 'content': 'Du bist ein Experte für österreichisches Steuerrecht. Beantworte präzise auf Deutsch. Nur die Antwort, keine Floskeln.'},
        {'role': 'user', 'content': str(row['prompt'])}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.1, do_sample=True, repetition_penalty=1.2, no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
    if len(ans) > 500: ans = ans[:497] + '...'
    pd.DataFrame([{'id': row['id'], 'answer': ans}]).to_csv(OUTPUT_PATH, mode='a', header=False, index=False)


In [ ]:
# ZELLE 6: Ergebnis prüfen
import pandas as pd
df_out = pd.read_csv(OUTPUT_PATH)
print(f'Zeilen: {len(df_out)}')
df_out.head(10)

In [ ]:
# ZELLE 7: Download der Output-CSV
from IPython.display import FileLink
print('Klicke den Link zum Download:')
FileLink(OUTPUT_PATH)